# Reaktive Maschinenagenten mit Mesa (v3.x)

In diesem Notebook wirst du:
- **Mesa 3.x** installieren,
- einen **reaktiven Agenten** implementieren, der die Temperatur einer Maschine überwacht,
- ein einfaches **Fabrikmodell** mit mehreren Maschinenagenten aufbauen,
- eine **Visualisierung mit Mesa 3.x** im Notebook oder Browser starten und verschiedene Parameter erkunden.


## 1. Installation
Zunächst installieren wir eine aktuelle Mesa-Version **ab 3.x** inklusive Visualisierungsunterstützung.


In [ ]:
#!pip install -U "mesa[viz]>=3.0"

## 2. Mesa-Grundlagen
Mesa-Modelle bestehen aus drei Hauptteilen:
- einer **Model**-Klasse, die den globalen Zustand verwaltet,
- einer oder mehreren **Agent**-Klassen, die das Verhalten einzelner Agenten definieren,
- einer optionalen **Visualisierung**, um das Modell während der Ausführung zu beobachten.

In diesem Beispiel bauen wir einen einfachen *reaktiven* Agenten:
- Er misst seine eigene Temperatur.
- Er wendet eine **Schwellenwert-Regel** an: Liegt die Temperatur über einem Grenzwert, wechselt der Agent in den Zustand `"HOT"`, andernfalls bleibt oder wechselt er in `"OK"`.


## 3. Implementierung des `MachineAgent`
Der `MachineAgent` repräsentiert eine einzelne Maschine in einer Fabrik. Er besitzt eine Temperatur und eine einfache reaktive Regel.


In [1]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Reaktiver Agent, der eine Maschine repräsentiert und deren Temperatur überwacht.

    Regel:
        if temperature > threshold -> state = 'HOT'
        else -> state = 'OK'
    """

    def __init__(self, model, threshold=70):
        super().__init__(model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"

    def sense_temperature(self):
        """Einfaches Sensormodell: Temperatur + zufälliges Rauschen."""
        if self.state != "HOT":
            noise = random.uniform(-2, 4)
            self.temperature = self.temperature + noise

    def decide(self):
        """Reaktive Entscheidungsregel, die nur auf der aktuellen Temperatur basiert."""
        if self.temperature > self.threshold:
            self.state = "HOT"
        else:
            self.state = "OK"

    def act(self):
        """Maschine abkühlen (z. B. Simulation einer Drosselung der Maschine)."""
        if self.state == "HOT":
            self.temperature = self.temperature - 10

    def step(self):
        """Agentenschritt = wahrnehmen -> entscheiden -> handeln."""
        self.sense_temperature()
        self.decide()
        self.act()


## 4. Implementierung des `FactoryModel`
Das Modell platziert mehrere Maschinen auf einem Gitter.
Außerdem verwenden wir einen **DataCollector**, um zu verfolgen, wie viele Maschinen sich im den Zustand `"HOT"` befinden.


In [2]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if a.state == "HOT")


class FactoryModel(Model):
    """Einfaches Fabrikmodell mit einem Gitter aus Maschinenagenten."""

    def __init__(self, width=10, height=10, density=0.3, threshold=70, seed=None):
        super().__init__(seed=seed)
        self.width = width
        self.height = height
        self.density = density
        self.threshold = threshold

        self.grid = MultiGrid(width, height, torus=False)

        for x in range(self.width):
            for y in range(self.height):
                if self.random.random() < self.density:
                    agent = MachineAgent(self, threshold=self.threshold)
                    self.grid.place_agent(agent, (x, y))

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines": count_hot_machines,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


## 5. Visualisierung mit Mesa 3.x
Mesa 3.x verwendet die neue Visualisierung über `SolaraViz`. Damit funktioniert die Darstellung deutlich besser in modernen Notebook- und Browser-Workflows als der alte Tornado-Server.

Die Schritte sind:
1. Eine **Darstellungsfunktion** (`agent_portrayal`) definieren.
2. Eine Modellinstanz oder Modellklasse mit Parametern an `SolaraViz` übergeben.
3. Die Visualisierung im Notebook anzeigen oder über `solara run` als App starten.

⚠️ **Hinweis:** Falls die Anzeige im Notebook nicht direkt funktioniert, speichere den Code in eine Python-Datei und starte ihn mit `solara run datei.py`.


In [16]:
from mesa.visualization import (
    SolaraViz,
    make_plot_component,
    make_space_component,
)

import matplotlib.pyplot as plt


# ============================================================
# Agentendarstellung
# ============================================================

def agent_portrayal(agent):

    return {
        "color": {
            "HOT": "red",
            "OK": "green",
        }.get(agent.state, "gray"),

        "size": 600,
        "marker": "s",
        "alpha": 0.9,
    }


# ============================================================
# Nachbearbeitung matplotlib
# ============================================================

def post_process(ax):

    # Figure-Größe
    ax.figure.set_size_inches(6, 6)

    # Legende entfernen
    legend = ax.get_legend()

    if legend:
        legend.remove()

    # Temperaturtexte ergänzen
    for agent in model_instance.agents:

        x, y = agent.pos

        ax.text(
            x,
            y,
            f"{agent.temperature:.1f}°C",

            ha="center",
            va="center",

            fontsize=7,
            color="white",
            fontweight="bold",
        )

    # Grid schöner
    ax.grid(True, alpha=0.3)


# ============================================================
# Space Component
# ============================================================

space_component = make_space_component(
    agent_portrayal,
    post_process=post_process,
)


# ============================================================
# Plot-Komponente
# ============================================================

plot_component = make_plot_component(
    {
        "HotMachines": "red",
    }
)


# ============================================================
# Modellparameter
# ============================================================

model_params = {
    "width": 10,
    "height": 10,

    "density": {
        "type": "SliderFloat",
        "value": 0.3,
        "label": "Dichte",
        "min": 0.1,
        "max": 1.0,
        "step": 0.1,
    },

    "threshold": {
        "type": "SliderInt",
        "value": 70,
        "label": "Temperatur-Schwelle",
        "min": 30,
        "max": 100,
        "step": 5,
    },
}


# ============================================================
# Modellinstanz
# ============================================================

model_instance = FactoryModel(
    width=10,
    height=10,
    density=0.3,
    threshold=70,
)


# ============================================================
# SolaraViz
# ============================================================

page = SolaraViz(
    model_instance,
    components=[
        space_component,
        plot_component,
    ],
    model_params=model_params,
    name="Reaktive Maschinenagenten",
)

page

Cannot show ipywidgets in text

## 6. Simulation ohne Visualisierung testen
Falls man nur die Modelllogik testen möchte, kann man das Modell auch direkt einige Schritte laufen lassen.


In [9]:
model = FactoryModel(width=10, height=10, density=0.3, threshold=70, seed=42)
for _ in range(20):
    model.step()

model.datacollector.get_model_vars_dataframe().tail()

,HotMachines
16,0
17,0
18,0
19,0
20,0


## 7. Erkundungsaufgaben (für das Labor)

Nutzen Sie das laufende Modell, um das Verhalten der reaktiven Agenten zu untersuchen:

1. **Ändern Sie den Schwellenwert**
   - Starten Sie mit `threshold=70` und testen Sie anschließend 60 oder 80.
   - Was passiert im Zeitverlauf mit der Anzahl der `"HOT"`-Maschinen?

2. **Ändern Sie die Dichte**
   - Erhöhen oder verringern Sie den Parameter `density`.
   - Wie beeinflusst dies den Gesamtzustand des Systems?

3. **Erweitern Sie die Zustandslogik (optional)**
   - Fügen Sie weitere Zustände hinzu, z. B. `"COOL"` oder `"WARNING"`, und ordnen Sie diesen unterschiedliche Farben in der Funktion `machine_portrayal` zu.
   - Definieren Sie eigene Temperaturbereiche für die jeweiligen Zustände.

4. **(Fortgeschritten) Wartungsagent hinzufügen**
   - Erstellen Sie einen zweiten Agententyp, der sich durch das Grid bewegt und „heiße“ Maschinen repariert, indem Temperatur bzw. Zustand zurückgesetzt werden.

5. **(Fortgeschritten) Auftragsagent hinzufügen**
   - Erstellen Sie einen dritten Agententyp, der Aufträge durch das Grid bewegt und selbstständig freie Maschinen findet.